# Replica of the scITD Analysis

The goal of this ntoebook is to replicate, on the breast-cancer data which is provided, the analysis which scITD conducts in order to gain insight on how tensor decompositions can be used to meaningfully analyse cell-cycle data.

## 0. Imports:

In [151]:
import numpy as np
import pandas as pd
import scanpy as sc

import tensorly as tl
from tensorly.decomposition import tucker

## 1. Data import:

In [137]:
adata = sc.read_h5ad("data/breast_cancer_zikry_wayne/44PE_merged.h5ad")

Understanding our data:

In [138]:
print(f'n_obs x n_vars: {adata.shape}')

n_obs x n_vars: (90594, 103)


In [139]:
print(f'cell barcodes (row index): {adata.obs_names}\n')
print(f'gene/feature names (column index): {adata.var_names}\n')

cell barcodes (row index): Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '6178-6', '6179-6', '6180-6', '6181-6', '6182-6', '6183-6', '6184-6',
       '6185-6', '6186-6', '6187-6'],
      dtype='object', length=90594)

gene/feature names (column index): Index(['cyto_mean_ERa', 'cyto_mean_Ki67', 'cyto_mean_PR', 'cyto_mean_DAPI',
       'cyto_mean_HES1', 'cyto_mean_p38', 'cyto_mean_p16', 'cyto_mean_PH3',
       'cyto_mean_p65', 'cyto_mean_pAKT',
       ...
       'nuclear_mean_bCat', 'nuclear_mean_SKP2', 'nuclear_mean_E2F1',
       'nuclear_mean_HER2', 'nuclear_mean_YAP1', 'nuclear_mean_CDT1',
       'nuclear_mean_pCdc6', 'nuclear_mean_cycE2', 'DNA_int', 'pRB/RB'],
      dtype='object', length=103)



In [140]:
adata.obs

,core,binucleated,CellID,GMM_phase_labels,DNA_int_obs,nuclear_mean_cycA2_obs,Endocycle,drug_name
0,1.0,0,0,G0,2.482551,585.705651,Diploid,Control
1,1.0,0,1,S,2.267976,371.223898,Diploid,Control
2,1.0,0,2,S,2.783111,1585.394977,Diploid,Control
3,1.0,0,3,G0,2.195485,111.103687,Diploid,Control
4,1.0,0,4,G1,2.199392,131.529825,Diploid,Control
...,...,...,...,...,...,...,...,...
6183-6,8.0,0,6183,S,2.211465,1546.934177,Diploid,Dinaciclib
6184-6,8.0,0,6184,G1,1.799833,147.419776,Diploid,Dinaciclib
6185-6,8.0,0,6185,G1,1.930990,148.797964,Diploid,Dinaciclib
6186-6,8.0,0,6186,G1,4.087362,114.891078,Polyploid,Dinaciclib


In [141]:
adata.obs['drug_name'].value_counts()

drug_name
RO-3306         22592
Palbociclib     17570
Talazoparib     14458
PF-3600          9202
CVT-313          7385
Control          7110
Dinaciclib       6188
Samuraciclib     6089
Name: count, dtype: int64

In [142]:
pd.DataFrame(adata.X, columns=adata.var_names, index=adata.obs_names)

,cyto_mean_ERa,cyto_mean_Ki67,cyto_mean_PR,cyto_mean_DAPI,cyto_mean_HES1,cyto_mean_p38,cyto_mean_p16,cyto_mean_PH3,cyto_mean_p65,cyto_mean_pAKT,...,nuclear_mean_bCat,nuclear_mean_SKP2,nuclear_mean_E2F1,nuclear_mean_HER2,nuclear_mean_YAP1,nuclear_mean_CDT1,nuclear_mean_pCdc6,nuclear_mean_cycE2,DNA_int,pRB/RB
0,33.236253,329.340122,164.185336,918.853360,110.411405,321.480652,454.437882,236.920570,281.370672,114.953157,...,232.508634,447.116954,142.361852,116.832025,163.726845,157.045526,2498.128728,1102.920722,2.482551,1.615529
1,213.103477,611.276490,1174.773179,460.025662,260.144040,1212.525662,440.814570,280.472682,253.663079,126.983444,...,486.495360,582.495360,167.380510,122.864269,286.300464,248.694896,3870.610209,2393.815545,2.267976,2.126723
2,188.183391,1297.884083,1215.052768,449.261246,246.387543,1336.935986,389.608131,321.261246,257.562284,131.964533,...,412.441781,1847.090183,170.787671,119.810502,281.863014,222.504566,3654.180365,2147.679224,2.783111,2.187872
3,370.501193,175.214399,1291.324582,415.553302,367.265712,1140.675418,540.842880,307.439539,278.285998,124.886635,...,514.115207,195.294931,178.384793,124.158986,291.124424,174.017281,4895.017281,2143.607143,2.195485,0.881578
4,76.006030,1751.752186,646.121495,255.706663,170.353633,1004.617124,309.799518,212.107627,228.789267,137.582153,...,310.319298,671.204094,137.091813,106.673099,172.815205,967.381287,3436.363743,1264.115789,2.199392,3.259080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6183-6,111.520810,998.093738,848.471691,274.190851,382.217473,1007.234346,508.085114,236.176978,246.051744,119.383577,...,364.302954,1056.803376,166.851477,111.979747,246.891139,195.589873,4277.421941,2075.454852,2.211465,3.205214
6184-6,98.404021,131.331589,532.146792,242.550812,178.311760,924.379785,270.480033,156.541724,192.607546,125.030019,...,312.280784,192.758396,140.451493,89.822761,192.987873,163.861007,2315.606343,1375.266791,1.799833,1.754812
6185-6,112.893740,166.720758,638.205107,252.177100,263.186985,854.596376,215.831137,181.151565,209.538715,111.624382,...,354.228661,374.604542,169.733751,103.410337,254.008614,961.241973,4045.628818,1875.848865,1.930990,3.141703
6186-6,69.801454,132.801246,568.855452,186.567186,290.068951,764.282035,196.247975,178.782555,212.201661,113.844444,...,248.651217,216.647740,148.087679,107.511008,2216.246041,507.805330,2431.222866,1393.193511,4.087362,2.432299


## 2. scITD-esque analysis:

### 2.1 Pseudobulking:
Our idea is to, for a first go, model our tensor via: $\textit{drug name} \times \textit{proteins} \times \textit{GMM phase}$. We thus aggregate $\texttt{adata.X}$ by $\textit{drug name} \times \textit{GMM phase}$. This will yield a data frame with dimensionality: $\lvert \{ \textit{drug name} \} \rvert \cdot \lvert \{ \textit{GMM phase} \} \rvert \times \lvert \{ \textit{proteins} \} \rvert = 40 \times 103$.

Rewording Mitchel et al. (2024): When $C$ cell-stages from $n$ drug types are collapsed into pseudobulk profiles, the dataset can be represented as a threedimensional matrix: a tensor $T$ with dimensions $n × P × C$, where $P$ is the number of proteins. In our case $C = 5$, $n = 8$, and $P = 103$.

In [143]:
# see form_tensor.R (205-245)
def get_pseudobulk(adata):
    df = adata.to_df()
    df['drug_name'] = adata.obs['drug_name'].values
    df['phase'] = adata.obs['GMM_phase_labels'].values
    return df.groupby(['phase', 'drug_name'], observed=True).mean()

In [144]:
pseudobulk_df = get_pseudobulk(adata)
pseudobulk_df

cyto_mean_ERa  cyto_mean_Ki67  cyto_mean_PR  \
phase drug_name                                                   
G0    CVT-313          144.403375      650.511004   1046.049668   
      Control          172.451072      455.401369   1063.037644   
      Dinaciclib        90.205817      268.852801    671.930215   
      PF-3600          174.445961      208.426434   1326.868225   
      Palbociclib      154.994526      634.704098   1097.190539   
      RO-3306          154.467887      819.984471    952.930116   
      Samuraciclib     158.189431      152.268561    857.346355   
      Talazoparib      168.460938      715.535168   1126.953571   
G1    CVT-313          174.206221      750.615114   1100.399208   
      Control          172.313717      885.251222   1075.313257   
      Dinaciclib       114.639855      425.161080    742.589387   
      PF-3600          198.539584      240.081171   1445.282922   
      Palbociclib      190.937620      583.075526   1171.386528   
      RO-3306          221.184972      996.427606   1060.656297   
      Samuraciclib     163.072132      193.862716    931.600047   
      Talazoparib      242.090615      566.512759   1307.685062   
G2    CVT-313          158.721790     2359.651114   1186.887234   
      Control          144.264792     2390.977230   1162.367809   
      Dinaciclib       105.028309     1444.125692    807.450749   
      PF-3600          176.197554     2265.711638   1535.851249   
      Palbociclib      166.421940     2281.990278   1270.865112   
      RO-3306          187.539553     3062.533142   1142.612376   
      Samuraciclib     114.005606     1181.435188    960.247727   
      Talazoparib      169.255457     2355.713045   1264.953912   
M     CVT-313          228.192392     3981.239329   1422.028830   
      Control          215.970166     4419.456473   1400.172644   
      Dinaciclib       159.181807     2320.132360    922.613648   
      PF-3600          273.307052     2575.646847   1920.634993   
      Palbociclib      309.354928     4008.331305   1555.327332   
      RO-3306          380.719360     4452.042030   1451.544029   
      Samuraciclib     179.354426     2276.178585   1177.066308   
      Talazoparib      262.842580     4031.383641   1623.338601   
S     CVT-313          150.435119     1410.625871   1133.801547   
      Control          145.254306     1487.983851   1010.050238   
      Dinaciclib        99.595067      866.599486    785.470630   
      PF-3600          155.962367     1275.603851   1385.180462   
      Palbociclib      157.341142     1353.448250   1178.212832   
      RO-3306          171.070500     1534.778796   1011.343726   
      Samuraciclib     116.360794      794.136991    905.957232   
      Talazoparib      174.419240     2519.956482   1256.313091   

                    cyto_mean_DAPI  cyto_mean_HES1  cyto_mean_p38  \
phase drug_name                                                     
G0    CVT-313           408.072220      258.124475    1192.351613   
      Control           416.187741      261.272853    1168.110701   
      Dinaciclib        283.607087      249.830882     840.668614   
      PF-3600           355.898766      310.288988    1633.678264   
      Palbociclib       492.411975      226.868631    1602.962035   
      RO-3306           451.569048      223.825998    1169.420685   
      Samuraciclib      272.967585      305.267242    1462.874836   
      Talazoparib       419.409333      250.409155    1248.542469   
G1    CVT-313           415.491355      259.243086    1318.937715   
      Control           383.017906      259.032814    1219.184502   
      Dinaciclib        282.697839      243.332474     971.906959   
      PF-3600           332.249983      312.795122    1842.600718   
      Palbociclib       421.256313      230.341434    1705.480797   
      RO-3306           485.867506      230.297833    1340.715380   
      Samuraciclib      274.823959      310.097282    1627.940894   
      Talazoparib       427.5989

Restating Mitchel et al. (2024): we scale and center in each cell phase across drugs.

In [145]:
def normalize_and_scale(pseudobulk_df):
    return pseudobulk_df.groupby('phase', observed=True).transform(lambda x: (x - x.mean()) / x.std())

In [146]:
pseudobulk_norm_df = normalize_and_scale(pseudobulk_df)
pseudobulk_norm_df

cyto_mean_ERa  cyto_mean_Ki67  cyto_mean_PR  \
phase drug_name                                                   
G0    CVT-313           -0.288307        0.640205      0.144933   
      Control            0.748538       -0.129418      0.232053   
      Dinaciclib        -2.291839       -0.865271     -1.773666   
      PF-3600            0.822283       -1.103627      1.585057   
      Palbociclib        0.103218        0.577854      0.407199   
      RO-3306            0.083750        1.308705     -0.332613   
      Samuraciclib       0.221325       -1.325146     -0.822796   
      Talazoparib        0.601034        0.896698      0.559833   
G1    CVT-313           -0.269027        0.590004     -0.018385   
      Control           -0.317903        1.055927     -0.134709   
      Dinaciclib        -1.807393       -0.536263     -1.677547   
      PF-3600            0.359409       -1.176752      1.580839   
      Palbociclib        0.163080        0.010216      0.310783   
      RO-3306            0.944250        1.440664     -0.202673   
      Samuraciclib      -0.556577       -1.336695     -0.801106   
      Talazoparib        1.484161       -0.047101      0.942798   
G2    CVT-313            0.204596        0.324893      0.094413   
      Control           -0.284917        0.377933     -0.018607   
      Dinaciclib        -1.613462       -1.225250     -1.654561   
      PF-3600            0.796324        0.165837      1.702927   
      Palbociclib        0.465322        0.193399      0.481500   
      RO-3306            1.180364        1.514993     -0.109667   
      Samuraciclib      -1.309492       -1.670030     -0.950259   
      Talazoparib        0.561265        0.318225      0.454253   
M     CVT-313           -0.318844        0.500060     -0.040644   
      Control           -0.488848        0.963164     -0.114291   
      Dinaciclib        -1.278740       -1.255381     -1.723477   
      PF-3600            0.308673       -0.985356      1.639463   
      Palbociclib        0.810077        0.528691      0.408519   
      RO-3306            1.802713        0.997600      0.058811   
      Samuraciclib      -0.998151       -1.301831     -0.866071   
      Talazoparib        0.163119        0.553052      0.637691   
S     CVT-313            0.159137        0.009919      0.260428   
      Control           -0.040475        0.156512     -0.377625   
      Dinaciclib        -1.799681       -1.021008     -1.535542   
      PF-3600            0.372096       -0.245947      1.556519   
      Palbociclib        0.425219       -0.098433      0.489409   
      RO-3306            0.954198        0.245188     -0.370956   
      Samuraciclib      -1.153714       -1.158324     -0.914321   
      Talazoparib        1.083221        2.112093      0.892088   

                    cyto_mean_DAPI  cyto_mean_HES1  cyto_mean_p38  \
phase drug_name                                                     
G0    CVT-313             0.264488       -0.081540      -0.368986   
      Control             0.368904        0.016761      -0.460750   
      Dinaciclib         -1.336910       -0.340488      -1.700270   
      PF-3600            -0.406788        1.547176       1.301640   
      Palbociclib         1.349624       -1.057431       1.185365   
      RO-3306             0.824129       -1.152430      -0.455791   
      Samuraciclib       -1.473801        1.390384       0.655070   
      Talazoparib         0.410354       -0.322433      -0.156278   
G1    CVT-313             0.502547       -0.125997      -0.414630   
      Control             0.068703       -0.132529      -0.767085   
      Dinaciclib         -1.271571       -0.620290      -1.640784   
      PF-3600            -0.609556        1.537700       1.435615   
      Palbociclib         0.579567       -1.023882       0.951133   
      RO-3306             1.442771       -1.025236      -0.337683   
      Samuraciclib       -1.376766        1.453886       0.677163   
      Talazoparib         0.6643

In [148]:
phases = pseudobulk_norm_df.index.get_level_values('phase').unique()
drugs = pseudobulk_norm_df.index.get_level_values('drug_name').unique()

np_tensor = np.stack(
    [pseudobulk_norm_df.xs(ph, level='phase').loc[drugs].values for ph in phases],
    axis=2
)

np_tensor.shape

(8, 103, 5)

### 2.2 Tucker decomposition:

In [153]:
tensor = tl.tensor(np_tensor)
core, factors = tucker(tensor, rank=[3, 10, 3])

In [154]:
factors[1]

array([[ 0.12537189,  0.11386294,  0.00539591, ...,  0.1119039 ,
        -0.06618373, -0.0473939 ],
       [ 0.03189943,  0.19689368, -0.00333134, ...,  0.0595155 ,
         0.07132969,  0.01628402],
       [ 0.16489949,  0.03556387, -0.0239627 , ...,  0.0108595 ,
        -0.0460032 ,  0.0537256 ],
       ...,
       [ 0.13792678, -0.1051307 , -0.06432658, ..., -0.04611838,
        -0.00616503,  0.00756898],
       [ 0.04532917, -0.04074751,  0.16966285, ...,  0.17305517,
        -0.14366353, -0.0056121 ],
       [-0.02987516,  0.08289876,  0.06351717, ...,  0.09454679,
         0.19655175, -0.02441366]], shape=(103, 10))

---